In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import random

spark = SparkSession.builder \
    .appName("Salting_AQE_Practice") \
    .getOrCreate()

# Create skewed dataset
data = []

# 90% of data has same key (skew)
for i in range(90000):
    data.append(("A", random.randint(1, 100)))

# Remaining distributed normally
for i in range(10000):
    data.append((f"K{i}", random.randint(1, 100)))

df_large = spark.createDataFrame(data, ["key", "value"])

# Small dimension table
dim_data = [("A", "Category_A")] + [(f"K{i}", f"Category_{i}") for i in range(10000)]
df_small = spark.createDataFrame(dim_data, ["key", "category"])

In [0]:
df_join = df_large.join(df_small, "key")
df_join.groupBy("category").count().show()

In [0]:
salt_range = 5

df_large_salted = df_large.withColumn(
    "salt",
    F.when(F.col("key") == "A", F.floor(F.rand() * salt_range))
     .otherwise(F.lit(0))
)

In [0]:
from pyspark.sql import Row

expanded_rows = []

for row in df_small.collect():
    if row["key"] == "A":
        for i in range(salt_range):
            expanded_rows.append((row["key"], row["category"], i))
    else:
        expanded_rows.append((row["key"], row["category"], 0))

df_small_salted = spark.createDataFrame(expanded_rows, ["key", "category", "salt"])

In [0]:
df_join_salted = df_large_salted.join(
    df_small_salted,
    ["key", "salt"]
)

df_join_salted.groupBy("category").count().show()

In [0]:
#spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")


In [0]:
spark.version

In [0]:
spark.conf.getAll()

In [0]:
spark.conf.get("spark.sql.adaptive.enabled")


In [0]:
df_large.join(df_small, "key").explain(True)